In [1]:
import os

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [4]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

100%|██████████| 9.91M/9.91M [00:11<00:00, 844kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 135kB/s]
100%|██████████| 1.65M/1.65M [00:04<00:00, 375kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 898kB/s]


In [6]:
x, labels = next(iter(train_loader))

print(x.shape)
print(labels.shape)

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [2]:
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )

        self.mu = nn.Linear(32, latent_dim)
        self.log_var = nn.Linear(32, latent_dim)

    def forward(self, x):
        h = self.shared(x)

        mu = self.mu(h)
        log_var = self.log_var(h)

        return mu, log_var

In [5]:
encoder = VAEEncoder(latent_dim=2)

x, labels = next(iter(train_loader))
x = x.view(x.size(0), -1)

mu, log_var = encoder(x)

print(mu.shape)
print(log_var.shape)

torch.Size([64, 2])
torch.Size([64, 2])


In [7]:
std = torch.exp(0.5 * log_var)
eps = torch.randn_like(std)
z = mu + eps * std

print("mu:", mu.shape)
print("std:", std.shape)
print("eps:", eps.shape)
print("z:", z.shape)

mu: torch.Size([64, 2])
std: torch.Size([64, 2])
eps: torch.Size([64, 2])
z: torch.Size([64, 2])


In [11]:
from torch import sigmoid


class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Linear(32, 128),
            nn.ReLU(),

            nn.Linear(128, 784),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.decoder(z)

In [12]:
decoder = VAEDecoder(latent_dim=2)

x_hat = decoder(z)

print(x_hat.shape)

torch.Size([64, 784])


In [13]:
from turtle import forward


class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()

        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)
        x_hat = self.decoder(z)

        return x_hat, mu, log_var

In [14]:
model = VAE(latent_dim=2)

x, labels = next(iter(train_loader))
x = x.view(x.size(0), -1)

x_hat, mu, log_var = model(x)

print("Reconstructed:", x_hat.shape)
print("Mu:", mu.shape)
print("Log variance:", log_var.shape)

Reconstructed: torch.Size([64, 784])
Mu: torch.Size([64, 2])
Log variance: torch.Size([64, 2])


In [17]:
def vae_loss(x_hat, x, mu, log_var):
    reconstruction_loss = nn.functional.binary_cross_entropy(
        x_hat,
        x,
        reduction="sum"
    )

    kl_loss = -0.5 * torch.sum(
        1 + log_var - mu.pow(2) - log_var.exp()
    )

    total_loss = reconstruction_loss + kl_loss

    return total_loss, reconstruction_loss, kl_loss

In [18]:
total, recon, kl = vae_loss(x_hat, x, mu, log_var)

print("Total loss:", total.item())
print("Reconstruction loss:", recon.item())
print("KL loss:", kl.item())

Total loss: 34870.97265625
Reconstruction loss: 34869.8671875
KL loss: 1.1050360202789307
